## Bronze to Silver: Data Cleansing and Transformation

In [0]:
from pyspark.sql.types import StringType, IntegerType, DateType, BooleanType
import pyspark.sql.functions as F
from delta.tables import DeltaTable

#### Create Widgets

In [0]:
dbutils.widgets.text("catalog_name", "ecommerce", "Catalog Name")
dbutils.widgets.text("storage_account_name", "ecommcistorage05", "Storage Account Name")
dbutils.widgets.text("container_name", "ecomm-raw-data", "Container Name")

In [0]:
catalog_name = dbutils.widgets.get("catalog_name")
storage_account_name = dbutils.widgets.get("storage_account_name")
container_name = dbutils.widgets.get("container_name")

print(catalog_name, storage_account_name, container_name)

ecommerce stgecomadlsddevap001 ecomm-raw-data


In [0]:
silver_checkpoint_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/checkpoint/silver/fact_order_items/"
print(silver_checkpoint_path)

abfss://ecomm-raw-data@stgecomadlsddevap001.dfs.core.windows.net/checkpoint/silver/fact_order_items/


In [0]:
df_batch = spark.read \
    .format("delta") \
    .table(f"{catalog_name}.bronze.brz_order_items")

display(df_batch.limit(10))

dt,order_ts,customer_id,order_id,item_seq,product_id,quantity,unit_price_currency,unit_price,discount_pct,tax_amount,channel,coupon_code,_rescued_data,ingest_timestamp,source_file
2024-01-10,2024-01-10T12:57:34.000Z,CUST000000078656,9344,1,2000000173474,Two,GBP,61,0%,15,web,PRIME5,null,2026-09-23T13:01:02.503Z,abfss://ecomm-raw-data@stgecomadlsddevap001.dfs.core.windows.net/order_items/landing/order_items_2024-01-10.csv
2024-01-10,2024-01-10T05:24:43.000Z,CUST000000023044,9345,1,2000000265544,Two,INR,45542,14%,3927,web,NEW10,null,2026-09-23T13:01:02.503Z,abfss://ecomm-raw-data@stgecomadlsddevap001.dfs.core.windows.net/order_items/landing/order_items_2024-01-10.csv
2024-01-10,2024-01-10T08:50:30.000Z,CUST000000251241,9346,1,2000000349015,Two,GBP,496,11%,45,app,FEST20,null,2026-09-23T13:01:02.503Z,abfss://ecomm-raw-data@stgecomadlsddevap001.dfs.core.windows.net/order_items/landing/order_items_2024-01-10.csv
2024-01-10,2024-01-10T08:50:30.000Z,CUST000000251241,9346,2,2000000465203,1,GBP,261,1%,47,app,FEST20,null,2026-09-23T13:01:02.503Z,abfss://ecomm-raw-data@stgecomadlsddevap001.dfs.core.windows.net/order_items/landing/order_items_2024-01-10.csv
2024-01-10,2024-01-10T02:16:38.000Z,CUST000000287680,9347,1,2000000054414,1,AUD,105,8%,5,app,PRIME5,null,2026-09-23T13:01:02.503Z,abfss://ecomm-raw-data@stgecomadlsddevap001.dfs.core.windows.net/order_items/landing/order_items_2024-01-10.csv
2024-01-10,2024-01-10T17:03:04.000Z,CUST000000071131,9348,1,2000000097275,1,INR,4011,0%,721,web,FEST20,null,2026-09-23T13:01:02.503Z,abfss://ecomm-raw-data@stgecomadlsddevap001.dfs.core.windows.net/order_items/landing/order_items_2024-01-10.csv
2024-01-10,2024-01-10T15:30:45.000Z,CUST000000010083,9349,1,2000000322919,1,AED,193,0%,35,app,null,null,2026-09-23T13:01:02.503Z,abfss://ecomm-raw-data@stgecomadlsddevap001.dfs.core.windows.net/order_items/landing/order_items_2024-01-10.csv
2024-01-10,2024-01-10T15:30:45.000Z,CUST000000010083,9349,2,2000000308517,1,AED,77,3%,9,app,null,null,2026-09-23T13:01:02.503Z,abfss://ecomm-raw-data@stgecomadlsddevap001.dfs.core.windows.net/order_items/landing/order_items_2024-01-10.csv
2024-01-10,2024-01-10T16:35:05.000Z,CUST000000202068,9350,1,2000000300672,1,AUD,32,13%,2,app,NEW10,null,2026-09-23T13:01:02.503Z,abfss://ecomm-raw-data@stgecomadlsddevap001.dfs.core.windows.net/order_items/landing/order_items_2024-01-10.csv
2024-01-10,2024-01-10T16:35:05.000Z,CUST000000202068,9350,2,2000000332666,1,AUD,20,10%,1,app,NEW10,null,2026-09-23T13:01:02.503Z,abfss://ecomm-raw-data@stgecomadlsddevap001.dfs.core.windows.net/order_items/landing/order_items_2024-01-10.csv


### Stream Bronze Table in a Dataframe

In [0]:
df = spark.readStream \
.format("delta") \
.table(f"{catalog_name}.bronze.brz_order_items")

### Perform Transformations and Cleaning

In [0]:
df = df.dropDuplicates(["order_id", "item_seq"])

# Transformation: Convert 'Two' → 2 and cast to Integer
df = df.withColumn(
    "quantity",
    F.when(F.col("quantity") == "Two", 2).otherwise(F.col("quantity")).cast("int")
)

# Transformation : Remove any '$' or other symbols from unit_price, keep only numeric
df = df.withColumn(
    "unit_price",
    F.regexp_replace("unit_price", "[$]", "").cast("double")
)

# Transformation : Remove '%' from discount_pct and cast to double
df = df.withColumn(
    "discount_pct",
    F.regexp_replace("discount_pct", "%", "").cast("double")
)

# Transformation : coupon code processing (convert to lower)
df = df.withColumn(
    "coupon_code", F.lower(F.trim(F.col("coupon_code")))
)

# Transformation : channel processing 
df = df.withColumn(
    "channel",
    F.when(F.col("channel") == "web", "Website")
    .when(F.col("channel") == "app", "Mobile")
    .otherwise(F.col("channel")),
)

#Transformation : Add processed time 
df = df.withColumn(
    "processed_time", F.current_timestamp()
)


In [0]:
#display(df.limit(20))

### Save to Silver Table

In [0]:
silver_checkpoint_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/checkpoint/silver/fact_order_items/"
print(silver_checkpoint_path)

abfss://ecomm-raw-data@stgecomadlsddevap001.dfs.core.windows.net/checkpoint/silver/fact_order_items/


In [0]:
def upsert_to_silver(microBatchDF, batchId):
    table_name = f"{catalog_name}.silver.slv_order_items"
    if not spark.catalog.tableExists(table_name):
        print("creating new table")
        microBatchDF.write.format("delta").mode("overwrite").saveAsTable(table_name)
        spark.sql(
            f"ALTER TABLE {table_name} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)"
        )
    else:
        deltaTable = DeltaTable.forName(spark, table_name)
        deltaTable.alias("silver_table").merge(
            microBatchDF.alias("batch_table"),
            "silver_table.order_id = batch_table.order_id AND silver_table.item_seq = batch_table.item_seq",
        ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()    

    



In [0]:
# This line is running a Structured Streaming job that:
# - Reads incremental data from Bronze (df).
# - For each batch → applies upsert_to_silver (update if exists, insert if not).
# - Writes into a Silver Delta table with schema evolution enabled.
# - Uses checkpointing for recovery.
# - Runs in batch-like mode (once or availableNow), not continuous streaming.

df.writeStream.trigger(availableNow=True).foreachBatch(
    upsert_to_silver
).format("delta").option("checkpointLocation", silver_checkpoint_path).option(
    "mergeSchema", "true"
).outputMode(
    "update"
).trigger(
    once=True
).start().awaitTermination()